### 1. Join hillslope, watershed, and channel stats to the pour points snapped gpkg using the 'WS_ID' field
    - Pour points snapped to the stream are necessary for SSN2 input
    - Prefixes 'ws_', 'hs_', and 'ch_' are added to fields for watersheds, hillslopes, and channels, respectively  

In [2]:
import os
import geopandas as gpd

def join_geopackages_on_wsid(points_file, watersheds_file, channels_file, hillslopes_file, output_file, overwrite = False):
    if overwrite and os.path.exists(output_file):
        os.remove(output_file)
    
    elif not overwrite and os.path.exists(output_file):
        print(f"{output_file} already exists. Set overwrite=True to overwrite.")
        return
    
    # Read the points geopackage
    points_gdf = gpd.read_file(points_file)

    # Read the other geopackages
    watersheds_gdf = gpd.read_file(watersheds_file)
    channels_gdf = gpd.read_file(channels_file)
    hillslopes_gdf = gpd.read_file(hillslopes_file)

    # Remove geometry columns from these datasets
    watersheds_df = watersheds_gdf.drop(columns='geometry')
    channels_df = channels_gdf.drop(columns='geometry')
    hillslopes_df = hillslopes_gdf.drop(columns='geometry')

    # Drop rows in channels_df that are not in hillslopes_df
    channels_df = channels_df[channels_df['WS_ID'].isin(hillslopes_df['WS_ID'])]

    # Drop rows in watersheds_df that are not in hillslopes_df
    watersheds_df = watersheds_df[watersheds_df['WS_ID'].isin(hillslopes_df['WS_ID'])]

    # Drop rows in points_gdf that are not in hillslopes_df
    points_gdf = points_gdf[points_gdf['WS_ID'].isin(hillslopes_df['WS_ID'])]
    
    # Rename columns in watersheds_df to add 'ws_' prefix, except 'WS_ID'
    ws_columns = {col: 'ws_' + col for col in watersheds_df.columns if col != 'WS_ID'}
    watersheds_df = watersheds_df.rename(columns=ws_columns)

    # Similarly for channels_df
    ch_columns = {col: 'ch_' + col for col in channels_df.columns if col != 'WS_ID'}
    channels_df = channels_df.rename(columns=ch_columns)

    # Similarly for hillslopes_df
    hs_columns = {col: 'hs_' + col for col in hillslopes_df.columns if col != 'WS_ID'}
    hillslopes_df = hillslopes_df.rename(columns=hs_columns)
    
    try:
        # Now perform the joins
        merged_gdf = points_gdf.merge(watersheds_df, on='WS_ID', how='left')
        merged_gdf = merged_gdf.merge(channels_df, on='WS_ID', how='left')
        merged_gdf = merged_gdf.merge(hillslopes_df, on='WS_ID', how='left')
    except:
        print("Hillslopes")
        print(hillslopes_df.columns)
        print("Channels")
        print(channels_df.columns)
        print("Watersheds")
        print(watersheds_df.columns)
        print("Points")
        print(points_gdf.columns)
        raise ValueError("The join failed. Check the column names in the input files.")

    print(f"Number of rows in the points dataset: {len(points_gdf)}")
    print(f"Number of rows in the watersheds dataset: {len(watersheds_df)}")
    print(f"Number of rows in the channels dataset: {len(channels_df)}")
    print(f"Number of rows in the hillslopes dataset: {len(hillslopes_df)}")
    print(f"Number of rows in the merged dataset: {len(merged_gdf)}")
    
    # Save the result to a new geopackage
    merged_gdf.to_file(output_file, driver='GPKG')


### Fill any null values in the points geopackge by using the median value for the three closest non-null neighbors 

In [3]:
def fill_null_values(gdf, output_gpkg=None):

    # Ensure the 'WS_ID' column exists and is numeric
    if 'WS_ID' not in gdf.columns:
        raise ValueError("The GeoDataFrame does not contain a 'WS_ID' column.")
    if not pd.api.types.is_numeric_dtype(gdf['WS_ID']):
        raise TypeError("'WS_ID' column must be numeric.")

    # Ensure the GeoDataFrame has a Coordinate Reference System (CRS) set
    if gdf.crs is None:
        print("Warning: The GeoDataFrame lacks a CRS. Proceeding without one.")

    # Sort the GeoDataFrame based on 'WS_ID'
    gdf = gdf.sort_values('WS_ID').reset_index(drop=True)

    # Identify features with null values in any attribute field (excluding 'geometry')
    features_with_nulls = gdf[gdf.drop(columns='geometry').isnull().any(axis=1)].copy()

    # Get number of null values in each field
    null_counts = features_with_nulls.isnull().sum()
    # filter out 0 null values
    null_counts = null_counts[null_counts > 0]
    print(f"Number of null values in original dataset\n: {null_counts}")
    # Get a list of WS_IDs as a NumPy array
    ws_ids = gdf['WS_ID'].values

    # Iterate over each feature with null values
    for idx, row in features_with_nulls.iterrows():
        current_ws_id = row['WS_ID']
        if pd.isnull(current_ws_id):
            continue  # Skip if WS_ID is null

        # Compute absolute differences between current WS_ID and all other WS_IDs
        ws_id_differences = np.abs(ws_ids - current_ws_id).astype(np.float64)
        ws_id_differences[idx] = np.inf  # Exclude the current feature

        # Sort indices based on WS_ID differences
        sorted_indices = np.argsort(ws_id_differences)

        # Initialize list to store indices of valid neighbors
        nearest_indices = []

        # List of fields that are null in the current row (excluding geometry)
        null_fields = row.drop(labels=['geometry']).index[row.drop(labels=['geometry']).isnull()].tolist()

        # Iterate over sorted indices to find up to 3 valid neighbors
        for neighbor_idx in sorted_indices:
            neighbor_row = gdf.iloc[neighbor_idx]

            # Check if neighbor has non-null values for all null fields of the current row
            neighbor_values = neighbor_row[null_fields]
            if neighbor_values.notnull().all():
                nearest_indices.append(neighbor_idx)
            if len(nearest_indices) >= 10:
                break  # Found 10 valid neighbors

        if not nearest_indices:
            print(f"WARNING: No valid neighbors found for WS_ID {current_ws_id}.")
            continue  # No valid neighbors to use for interpolation

        closest_features = gdf.iloc[nearest_indices]

        # Update null fields using median (numeric) or mode (categorical) of nearest features
        for field in null_fields:
            values = closest_features[field].dropna()
            if not values.empty:
                if pd.api.types.is_numeric_dtype(gdf[field]):
                    # Numeric field: use median
                    median_value = values.median()
                    gdf.at[idx, field] = median_value
                else:
                    # Categorical field: use mode
                    mode_value = values.mode()
                    if not mode_value.empty:
                        gdf.at[idx, field] = mode_value.iloc[0]

    # Save the updated GeoDataFrame
    if output_gpkg is not None:
        gdf.to_file(output_gpkg, driver='GPKG')
    return gdf

def drop_null_response_rows(gdf):
    """
    Reads a layer from the input GeoPackage, drops rows that contain any NULL values,
    and writes the result to the output GeoPackage.
    
    Parameters
    ----------
    input_gpkg : str
        Path to the input GeoPackage.
    input_layer : str
        Name of the layer to process.
    output_gpkg : str
        Path to the output GeoPackage (will be created or overwritten).
    """
    # Read the specified layer from the input GeoPackage
    null_response_rows = [col for col in gdf.columns if 'erosion' in col.lower() or 'deposition' in col.lower() or 'sfm' in col.lower()]
    gdf_clean = gdf.dropna(subset=null_response_rows, how='any')

    return gdf_clean

def remove_outliers(gdf, outcome_var, factor=2, output_gpkg=None):
    """
    Removes outliers from the GeoDataFrame based on the specified outcome variable
    using the IQR (Interquartile Range) method.

    Parameters
    ----------
    gdf : GeoDataFrame
        The input GeoDataFrame.
    outcome_var : str
        The column name from which to remove outliers. This column must be numeric.
    factor : float, optional
        The multiplier for the IQR to define the outlier bounds. Default is 2.
    output_gpkg : str, optional
        If provided, the filtered GeoDataFrame is written to this GeoPackage.

    Returns
    -------
    GeoDataFrame
        A GeoDataFrame with outliers removed based on the specified outcome variable.
    """
    import pandas as pd

    # Ensure the outcome variable exists and is numeric
    if outcome_var not in gdf.columns:
        raise ValueError(f"The GeoDataFrame does not contain the column '{outcome_var}'.")
    if not pd.api.types.is_numeric_dtype(gdf[outcome_var]):
        raise TypeError(f"The column '{outcome_var}' must be numeric.")

    # Calculate the first and third quartiles and the IQR
    Q1 = gdf[outcome_var].quantile(0.25)
    Q3 = gdf[outcome_var].quantile(0.75)
    IQR = Q3 - Q1

    # Define lower and upper bounds for outlier detection
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR

    # Filter out rows with outlier values in the outcome variable
    gdf_filtered = gdf[(gdf[outcome_var] >= lower_bound) & (gdf[outcome_var] <= upper_bound)]
    
    removed_count = len(gdf) - len(gdf_filtered)
    print(f"Removed {removed_count} / {len(gdf)} rows based on outlier detection in '{outcome_var}'")
    print(f"Upper bound: {upper_bound}, Lower bound: {lower_bound}")
    # Save the filtered GeoDataFrame if an output geopackage path is provided
    if output_gpkg is not None:
        gdf_filtered.to_file(output_gpkg, driver='GPKG')

    return gdf_filtered

def standardize_gdf_fields(gdf, exclude_columns=['WS_ID', 'geometry']):
    import geopandas as gpd
    from sklearn.preprocessing import RobustScaler


    # Define columns to exclude from scaling
    exclude_columns = ['WS_ID', 'geometry']
    columns_to_exclude = [col for col in gdf.columns if 'erosion' in col.lower() or 
                        'deposition' in col.lower() or 'sfm' in col.lower() or 
                        col == gdf.geometry.name or
                        col in exclude_columns]

    # Identify numeric columns, excluding specified columns
    numeric_cols = gdf.select_dtypes(include='number').columns
    columns_to_standardize = [col for col in numeric_cols if col not in columns_to_exclude]

    print(f"Excluding columns from standardization: {columns_to_exclude}")
    # Apply RobustScaler to the selected columns
    scaler = RobustScaler()
    gdf[columns_to_standardize] = scaler.fit_transform(gdf[columns_to_standardize])

    return gdf

def log_transformation(gdf, log_trans_outcome_vars = False):
    import geopandas as gpd
    import numpy as np

    # if log_trans_outcome_vars is True, log-transform all erosion, deposition, and SfM columns
    if log_trans_outcome_vars:
        log_transform_columns = [
            col for col in gdf.columns 
            if ('erosion' in col.lower() or 'deposition' in col.lower() or 'sfm' in col.lower() or 
                col in [
                    'ch_channel_width', 'ch_area', 'ch_valley_width', 'ch_slope over width', 
                    'ch_stream power', 'ch_channel width over valley width', 'hs_area', 
                    'hs_hillslope length', 'hs_flow accumulation max', 'hs_slope median'
                ])
        ]
    else:
        log_transform_columns = [
            'ch_channel_width', 'ch_area', 'ch_valley_width', 'ch_slope over width', 
            'ch_stream power', 'ch_channel width over valley width', 'hs_area', 
            'hs_hillslope length', 'hs_flow accumulation max', 'hs_slope median'
        ]
        print(f"Log-transforming columns: {log_transform_columns}")
    # Apply log transformation to the specified columns, handling zeros and negative values
    for col in log_transform_columns:
        #check for zero values and add a small constant to avoid log(0)
        if (gdf[col] <= 0).any():
            # add small constant to avoid log(0)
            gdf[col] += 1e-12
            #take absolute value
            gdf[col] = np.abs(gdf[col])
            gdf[col] = np.log1p(gdf[col])
        else:
            gdf[col] = np.log1p(gdf[col])  # log1p handles zero values by computing log(1 + x)

    # Save to GeoPackage
    return gdf

def make_positive(gdf, columns, epsilon=1e-10):
    """
    Ensures that specified columns in the GeoDataFrame are positive and non-zero.

    Parameters:
    - gdf (GeoDataFrame): The GeoDataFrame to modify.
    - columns (list of str): List of column names to process.
    - epsilon (float, optional): A small constant to add to each value to prevent zeros. Default is 1e-10.

    Returns:
    - GeoDataFrame: The modified GeoDataFrame with positive, non-zero values in specified columns.
    """
    # Take the absolute value of the specified columns
    #Filter out columns that are not in the dataframe
    columns = [col for col in columns if col in gdf.columns]
    
    gdf[columns] = gdf[columns].abs()
    
    # Add a small epsilon to prevent any zero values
    gdf[columns] += epsilon
    
    return gdf

def watersheds_to_numeric(gdf):
    # replace text values with numeric values
    text_to_numeric_mapping = {
        'LM2': 1,
        'LPM': 2,
        'MM': 3,
        'MPM': 4,
        'UM1': 5,
        'UM2': 6,
        'ME': 7,
        'MM': 8,
        'MW': 9,
        'UE': 10,
        'UM': 11,
        'UW': 12
    }

    # Assuming 'WS_ID' is the column to be replaced
    gdf['ch_watershed'] = gdf['ch_watershed'].replace(text_to_numeric_mapping)
    return gdf

def combine_watersheds_to_single_file(gpkg_paths, output_gpkg):
    """
    Appends point features from multiple GeoPackages into a single GeoPackage,
    ensuring that all input GeoPackages have the same CRS.

    Parameters:
        gpkg_paths (list of str): Paths to input GeoPackages.
        output_gpkg (str): Path to the output GeoPackage where the combined result will be written.

    Returns:
        None
        
    """
    import geopandas as gpd
    import os

    # Initialize an empty GeoDataFrame for the combined result
    combined_gdf = gpd.GeoDataFrame()
    crs = None  # To store the CRS of the first GeoPackage

    for gpkg_path in gpkg_paths:
        if not os.path.exists(gpkg_path): 
            print(f"GeoPackage {gpkg_path} does not exist, skipping...")
            continue

        # Read the GeoPackage into a GeoDataFrame
        gdf = gpd.read_file(gpkg_path)

        # Check CRS consistency
        if crs is None:
            crs = gdf.crs  # Set the CRS from the first GeoPackage
        else:
            if gdf.crs != crs:
                raise ValueError(f"CRS mismatch detected in {gpkg_path}: Expected {crs}, but got {gdf.crs}")

        # Append the GeoDataFrame to the combined_gdf
        combined_gdf = combined_gdf.append(gdf, ignore_index=True)

    # Set the CRS of the combined GeoDataFrame to the common CRS
    if crs is not None:
        combined_gdf.set_crs(crs, inplace=True)
    else:
        print("No valid GeoPackages found to combine.")
        return

    #convert output crs to EPSG 26913
    combined_gdf = combined_gdf.to_crs(epsg=26913)
    
    combined_gdf.to_file(output_gpkg, driver="GPKG", if_exists="replace")
    print(f"Combined features written to {output_gpkg}")
    return output_gpkg


### Preprocessing for normalized SfM erosion 

In [52]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd

def preprocess_watersheds_into_ssn( watershed_list, raw_ssn_points_dir, 
                                   ssn_points_output_dir, exclude_columns=[], log_trans_outcome_vars=False,
                                   outcome_var = 'sfm erosion',  # 'sfm erosion', 'sfm deposition', 'lidar erosion', 'lidar deposition',
                                   combine_watersheds = False, region = 'ETF'):
    
    if 'net' in outcome_var:
        outcome_var_name = 'sfm net change'
    else:
        # Nake outcome_var_name equal to text before the last space in outcome_var
        outcome_var_name = outcome_var.rsplit(' ', 1)[0]
    if combine_watersheds:
        # Combine individual GeoPackages into a single GeoPackage
        gpkg_list = [
            os.path.join(raw_ssn_points_dir, f"{watershed} ssn points.gpkg")
            for watershed in watershed_list
        ]
        raw_ssn_points_input = os.path.join(raw_ssn_points_dir, f"{region} ssn points.gpkg")
        os.makedirs(os.path.dirname(raw_ssn_points_input), exist_ok=True)
        combine_watersheds_to_single_file(gpkg_list, output_gpkg=raw_ssn_points_input)
        ssn_points_output = os.path.join(
            ssn_points_output_dir,
            "Combined Watersheds",
            f"{region} {outcome_var_name} ssn points.gpkg"
        )
        watershed_list = [region]
    
    for watershed in watershed_list:
        if not combine_watersheds:
            raw_ssn_points_input = os.path.join(raw_ssn_points_dir, f"{watershed} ssn points.gpkg")
            ssn_points_output = os.path.join(
                ssn_points_output_dir,
                "Individual Watersheds",
                f"{watershed} {outcome_var_name} ssn points.gpkg"
            )
        print(f"Processing {raw_ssn_points_input}...")
        # Load and preprocess the GeoDataFrame
        gdf = gpd.read_file(raw_ssn_points_input)
        
        
        unwanted_columns = ['ch_ch_area', 'flow accum max', 'elevation min', 'ch_lidar net change', 'ch_lidar net change norm','ch_elevation mean',
                            'ch_distance upstream', 'ch_distance downstream', 'ch_slope upstream', 'ch_slope downstream', 'ch_sfm net change norm']
            
        # Drop columns that are not needed
        for col in unwanted_columns:
            if col in gdf.columns:
                gdf = gdf.drop(columns=col)
        
        # Identify erosion and deposition related columns
        erosion_deposition_columns = [
            col for col in gdf.columns
            if 'erosion' in col.lower() or 'deposition' in col.lower() or 'sfm' in col.lower() or 'lidar' in col.lower()
        ]
        exclude_columns_extended = exclude_columns + erosion_deposition_columns

        #make exclude columns extended include any columns that contain words in the exclude_columns list
        exclude_columns_extended += [col for col in gdf.columns if any(word in col for word in exclude_columns)] 
        
        # Delete unnecessary columns
        
        gdf = gdf.drop(columns=[col for col in gdf.columns if 'sfm_erosion' in col.lower()])
        gdf = gdf.drop(columns=[col for col in gdf.columns if 'sfm_deposition' in col.lower()])
        gdf = gdf.drop(columns=[col for col in gdf.columns if 'sfm_net' in col.lower()])
        gdf = gdf.drop(columns=[col for col in gdf.columns if 'lidar_erosion' in col.lower()])
        gdf = gdf.drop(columns=[col for col in gdf.columns if 'lidar_deposition' in col.lower()])
        
        if 'sfm erosion' in outcome_var:
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'deposition' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'net' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'lidar' in col.lower()])
            outcome_var_column = [col for col in gdf.columns if outcome_var in col.lower() and 'sum' in col.lower()][0]

        elif 'sfm deposition' in outcome_var:
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'erosion' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'net' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'lidar' in col.lower()])
            outcome_var_column = [col for col in gdf.columns if outcome_var in col.lower() and 'sum' in col.lower()][0]
        elif 'lidar erosion' in outcome_var:
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'deposition' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'net' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'sfm' in col.lower()])
            outcome_var_column = [col for col in gdf.columns if outcome_var in col.lower() and 'sum' in col.lower()][0]
        elif 'sfm net change' in outcome_var:
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'erosion' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'deposition' in col.lower()])
            gdf = gdf.drop(columns=[col for col in gdf.columns if 'lidar' in col.lower()])
            outcome_var_column = [col for col in gdf.columns if outcome_var in col.lower()][0]

        if outcome_var_column not in gdf.columns:
            print(f"Available columns: {gdf.columns}")
            raise ValueError(f"Outcome variable column '{outcome_var_column}' not found in the GeoDataFrame.")
        # Continue with existing preprocessing steps
        
        print(f"Max of {outcome_var_name}: {gdf[outcome_var_column].max()}")
        print(f"Min of {outcome_var_name}: {gdf[outcome_var_column].min()}")
        gdf = drop_null_response_rows(gdf)
        gdf = fill_null_values(gdf)
        gdf = remove_outliers(gdf, outcome_var=outcome_var_column)
        gdf = log_transformation(gdf, log_trans_outcome_vars=log_trans_outcome_vars)
        gdf = standardize_gdf_fields(gdf, exclude_columns=exclude_columns_extended)
        gdf = make_positive(gdf, erosion_deposition_columns)
        gdf = watersheds_to_numeric(gdf)
        

        print(f"Mean of {outcome_var_column}: {gdf[outcome_var_column].mean()}")
        
        print(f"Saving {watershed} to {ssn_points_output}\n")
        os.makedirs(os.path.dirname(ssn_points_output), exist_ok=True)
        if os.path.exists(ssn_points_output):
            os.remove(ssn_points_output)
        gdf.to_file(ssn_points_output, driver='GPKG')


## CPF Watersheds

In [47]:
ssn_points_out_dir = r"CPF\Inputs"

spacing_list = [
    5, 10, 20]

region = 'CPF'

watersheds = [
    'ME', 
    'MM', 
    'MW', 
    'UE', 
    'UM', 
    'UW',
]
overwrite = True


## ETF Watersheds

In [44]:
import os 
import geopandas as gpd
import warnings
warnings.filterwarnings("ignore")


ssn_points_out_dir = r"ETF\Inputs"

spacing_list = [
    5, 
    10, 
    20
    ]
region = 'ETF'

watersheds = [
    'LM2',
    'LPM',
    'MM'
]
overwrite = True


In [53]:
log_trans_outcome_vars = False # Whether to log-transform erosion, deposition, and net change variables

for spacing in spacing_list:
    lidar_watersheds = [w for w in watersheds if w not in ['ME', 'MW']]
    space_out_dir = f'{ssn_points_out_dir}\\Segmented {spacing}m'
    raw_ssn_points_dir = os.path.join(space_out_dir, "Raw Data") 
    os.makedirs(space_out_dir, exist_ok=True)
    
    exclude_columns=['geometry', 'count', 'resolution'] # columns to exclude from standard scaling

    print(f"Processing {watershed} with {spacing}m spacing...")

    # # Individual watersheds SfM
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                         outcome_var='sfm erosion sum')
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                         outcome_var='sfm deposition sum')
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                      outcome_var='sfm net change')
    # Individual watersheds Lidar
    preprocess_watersheds_into_ssn(lidar_watersheds, raw_ssn_points_dir, space_out_dir, 
                        exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
                            outcome_var='lidar erosion sum')

    # # Combined watersheds SfM
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                      outcome_var='sfm erosion sum', combine_watersheds=True, region=region)
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                      outcome_var='sfm deposition sum', combine_watersheds=True, region=region)
    # preprocess_watersheds_into_ssn(watersheds, raw_ssn_points_dir, space_out_dir, 
    #                     exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
    #                      outcome_var='sfm net change', combine_watersheds=True, region=region)
    # # Combined watersheds Lidar
    preprocess_watersheds_into_ssn(lidar_watersheds, raw_ssn_points_dir, space_out_dir, 
                        exclude_columns=exclude_columns, log_trans_outcome_vars=log_trans_outcome_vars, 
                         outcome_var='lidar erosion sum', combine_watersheds=True, region=region)


Processing UW with 5m spacing...
Processing CPF\Inputs\Segmented 5m\Raw Data\MM ssn points.gpkg...
Max of lidar erosion: 0.0
Min of lidar erosion: -0.1703097656250125
Number of null values in original dataset
: ch_valley_width                       2
ch_channel width over valley width    2
ch_central slope difference           2
ch_change in slope over width         2
dtype: int64
Removed 5 / 214 rows based on outlier detection in 'ch_lidar erosion sum'
Upper bound: 0.06616679687500485, Lower bound: -0.1172716796875086
Log-transforming columns: ['ch_channel_width', 'ch_area', 'ch_valley_width', 'ch_slope over width', 'ch_stream power', 'ch_channel width over valley width', 'hs_area', 'hs_hillslope length', 'hs_flow accumulation max', 'hs_slope median']
Excluding columns from standardization: ['WS_ID', 'ch_lidar erosion mean', 'ch_lidar erosion count', 'ch_lidar erosion sum', 'geometry']
Mean of ch_lidar erosion sum: 0.02733914296782497
Saving MM to CPF\Inputs\Segmented 5m\Individual Wa

## Histogram of variables

In [ ]:
ws = r"Y:\ATD\GIS\ETF\Watershed Stats\SSN2\Inputs\Segmented 20m\Combined Watersheds\ET sfm erosion ssn points.gpkg"
column = 'ch_sfm erosion mean'
plot_all = True # plot histogram for all fields in gpkg 
title = "Histogram of ET normalized channel erosion"
# Plot using Seaborn with KDE
def plot_histogram_seaborn(gdf, column, bins=30, title=None, xlabel=None, ylabel='Frequency', kde=False):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import numpy as np
    data = gdf[column].dropna()
    # transform to log scale
    #data = np.log1p(data)
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))
    sns.histplot(data, bins=bins, kde=kde, color='skyblue', edgecolor='black')
    plt.title(title if title else f'Histogram of {column}', fontsize=16)
    plt.xlabel(xlabel if xlabel else column, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.show()
    
    # # perform 1/data^-2 transformation
    # data = -1/(data)
    # sns.set(style="whitegrid")
    # plt.figure(figsize=(10, 6))
    # sns.histplot(data, bins=bins, kde=kde, color='skyblue', edgecolor='black')
    # plt.title(title if title else f'Histogram of log({column})', fontsize=18)
    # plt.xlabel(xlabel if xlabel else column, fontsize=12)
    # plt.ylabel(ylabel, fontsize=12)
    # plt.show()

gdf = gpd.read_file(ws)

#standardize the ch_watershed column


if column in gdf.columns:
    plot_histogram_seaborn(
        gdf,
        column=column,
        bins=30,
        xlabel=r'Normalized Erosion (m$^{3}$/m$^{2}$)',
        ylabel='Number of Channel Segments',
        kde=True,
        title=title
    )


else:
    print(f"Column '{column}' not found in the GeoDataFrame.")
    print(f"Available columns: {gdf.columns}")

if plot_all:
    for column in gdf.columns:
        print(f"Processing column: {column}")
        # check if column is numeric
        import pandas as pd
        if pd.api.types.is_numeric_dtype(gdf[column]):
            plot_histogram_seaborn(
                gdf,
                column=column,
                bins=20,
                xlabel=f'{column}',
                ylabel='Number of Watersheds',
                kde=True
            )


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import gamma

ws = r"Y:\ATD\GIS\ETF\Watershed Stats\SSN2\Inputs\Combined Watersheds\ET erosion ssn points.gpkg"
column = 'ch_sfm erosion norm'
plot_all = False  # plot histogram for all fields in gpkg 
title = "Histogram of ET normalized channel erosion"

# Plot using Seaborn with Gamma distribution overlay
def plot_histogram_seaborn(gdf, column, bins=30, title=None, xlabel=None, ylabel='Frequency', kde=False):
    data = gdf[column].dropna()
    
    # Ensure data is positive for Gamma distribution
    if (data <= 0).any():
        raise ValueError("Gamma distribution can only be fitted to positive data.")
    
    # Fit Gamma distribution to the data
    shape, loc, scale = gamma.fit(data, floc=0)  # Fix location to 0 for better fit
    x = np.linspace(data.min(), data.max(), 1000)
    pdf_fitted = gamma.pdf(x, shape, loc, scale)
    
    # Calculate bin width for scaling the PDF
    bin_width = (data.max() - data.min()) / bins
    pdf_scaled = pdf_fitted * len(data) * bin_width
    
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))
    
    # Plot histogram
    sns_hist = sns.histplot(data, bins=bins, kde=kde, color='skyblue', edgecolor='black', label='Histogram')
    
    # Plot Gamma PDF
    plt.plot(x, pdf_scaled, color='red', lw=2, label='Gamma Fit')
    
    plt.title(title if title else f'Histogram of {column}', fontsize=16)
    plt.xlabel(xlabel if xlabel else column, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.legend()
    # add KDE to legend
    if kde:
        sns_hist.legend(['KDE', 'Gamma Fit', 'Histogram',])
    plt.show()

# Read the GeoDataFrame
gdf = gpd.read_file(ws)

if column in gdf.columns:
    plot_histogram_seaborn(
        gdf,
        column=column,
        bins=30,
        xlabel=r'Normalized Erosion (m$^{3}$/m$^{2}$)',
        ylabel='Number of Channel Segments',
        kde=True,
        title=title
    )
else:
    print(f"Column '{column}' not found in the GeoDataFrame.")
    print(f"Available columns: {gdf.columns}")

if plot_all:
    for column in gdf.columns:
        print(f"Processing column: {column}")
        # Check if column is numeric
        if pd.api.types.is_numeric_dtype(gdf[column]):
            plot_histogram_seaborn(
                gdf,
                column=column,
                bins=20,
                xlabel=f'{column}',
                ylabel='Number of Watersheds',
                kde=True
            )
